# Number of visits per pointing in each filter (u, g, r, i, z, y), per survey type: MAF NVisits metric

This notebook is the same as `10bis_FilterBalance`, but the number of visits per pointing is computed with the MAF metric `maf.CountMetric("observationStartMJD", metric_name="NVisits")` on a `maf.HealpixSlicer(nside=64)`, as in cell 9 of `03_fbs5.3.6/Footprint_v2026-10-07.ipynb`, instead of the own KD-tree counting of `10bis`. It is meant to check how the number of visits per pointing is obtained with MAF.

- creation date: 2026-11-22
- companion notebooks: `10bis_FilterBalance` (own counting), `10_FilterBalance` (filter balance versus time), `03_fbs5.3.6/Footprint_v2026-10-07` (cell 9: NVisits with MAF)
- survey modes definition: https://github.com/lsst/survey-strategy_lsst_io

## Goal and outputs

For each survey category, the number of visits received by each pointing is histogrammed, **one histogram per filter, with the usual filter colours** (same colour code as in `10_FilterBalance` and `10bis_FilterBalance`). The legend gives, for each filter, the **mean** and the **median** number of visits per pointing.

**Figures** (one panel per category):

1. **Figure 1**: non-stacked histograms (the six filters are overlaid);
2. **Figure 2**: stacked histograms (the six filters are stacked, the total height is the number of pointings).

**Steps**

1. Region labels (scheduler map), loading of the visits and boolean masks of the categories (same as `10bis_FilterBalance`).
2. Number of visits per pointing, per filter and per category, with the MAF `NVisits` metric; check against cell 9 of `Footprint_v2026-10-07`.
3. Table of the mean and median number of visits per pointing (`.csv`).
4. Figures 1 and 2.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) and the MAF maps (`.npz`) go to `DATA10ter_FilterBalance/`; all figures (`.png` and `.pdf`) go to `FIG10ter_FilterBalance/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
import rubin_sim.maf as maf

from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,  # general size
        "axes.labelsize": 16,  # X and Y labels
        "axes.titlesize": 16,  # axes title
        "xtick.labelsize": 14,  # X ticks
        "ytick.labelsize": 14,  # Y ticks
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 14,
    }
)

In [ ]:
# Output directories: tables and MAF maps go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA10ter_FilterBalance"
FIG_DIR = "FIG10ter_FilterBalance"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by another path to analyse another run.
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

BANDS = list("ugrizy")
# Usual Rubin colours of the six filters (same as in 10_FilterBalance)
BAND_COLORS = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
NSIDE = 64  # HEALPix resolution of the scheduler map (region labels)

MIN_VISITS_TOTAL = 1000  # below this, the category is removed from the study

# Number of visits per pointing from MAF (see section 2)
NSIDE_POINTING = 64  # resolution of the MAF HealpixSlicer (64 in cell 9 of Footprint_v2026-10-07)
USE_CACHE = True  # read the NVisits maps from DATA_DIR if already computed (delete the .npz to recompute)
RUN_CELL9_CHECK = True  # also run MAF exactly as in cell 9 (MetricBundleGroup) and compare

# Histograms
N_BINS = 50  # maximum number of bins of a histogram
LOG_Y = False  # True: logarithmic scale for the number of pointings
LOG_X_CATEGORIES = ["All visits"]  # categories drawn with logarithmic bins and a logarithmic x axis
# x range of the histograms: category -> (xmin, xmax), None = automatic
XLIM = {}  # for instance {"WFD": (0, 300), "DDF": (None, 5000)}
XMAX_PERCENTILE = 99.5  # automatic xmax: percentile of the counts of the category (None = maximum)

# Region definition: name -> list of labels of the scheduler map.
# 'euclid_overlap' is merged with 'lowdust' (as in 02_SurveyAreas and 07).
WFD_REGIONS = {
    "low dust": ["lowdust", "euclid_overlap"],
    "bulge": ["bulgy"],
    "virgo": ["virgo"],
    "SMC-LMC": ["LMC_SMC"],
}
MINI_SURVEYS = {
    "NES": ["nes"],
    "SCP": ["scp"],
    "MW non-WFD": ["dusty_plane"],
}

## 1. Region labels, loading of the visits and categories

Same code as in `10bis_FilterBalance` (and `10_FilterBalance`). `CurrentAreaMap` returns one survey-mode label per HEALPix pixel; a label of the map not listed in `WFD_REGIONS` or `MINI_SURVEYS` is automatically added as a mini-survey. The micro-surveys are selected from `scheduler_note` (priority ToO > TwilightNEO > RGES > DDF); the other visits are labelled by the pixel containing the visit centre. The column `rotSkyPos` is also read, because the MAF slicer needs it.

Each visit is counted once in `WFD`, `DDF`, `Mini-surveys (total)`, `Micro-surveys without DDF (total)` (check below). `WFD low dust` is a subset of `WFD`, and `All visits` is the whole survey.

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)
label_names = [l for l in np.unique(labels) if l != ""]
print("Labels in the target map:", label_names)

# Labels configured above but absent from the map
for name, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items():
    for l in labs:
        if l not in label_names:
            print(f"WARNING: label {l} of region {name} is not in the target map")

# Labels of the map not configured above: added as mini-surveys
used_labels = {l for labs in list(WFD_REGIONS.values()) + list(MINI_SURVEYS.values()) for l in labs}
for l in label_names:
    if l not in used_labels:
        print(f"Label {l} is not in WFD_REGIONS or MINI_SURVEYS: added as a mini-survey")
        MINI_SURVEYS[l] = [l]

mini_names = list(MINI_SURVEYS)
wfd_labels = [l for labs in WFD_REGIONS.values() for l in labs]
mini_labels = [l for labs in MINI_SURVEYS.values() for l in labs]

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, rotSkyPos, "
    "visitExposureTime, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits = visits.sort_values("observationStartMJD").reset_index(drop=True)
N_ALL = len(visits)
print(f"{N_ALL} visits in the survey")

# Micro-surveys from scheduler_note (one tag per visit)
note = visits["scheduler_note"].fillna("")
is_too = note.str.contains("ToO").values
is_twilight = note.str.contains("twilight_near_sun").values
is_rges = note.str.contains("RGES").values
is_ddf = note.str.contains("DD").values & ~is_rges
visits["micro"] = np.select(
    [is_too, is_twilight, is_rges, is_ddf],
    ["ToO", "TwilightNEO", "RGES", "DDF"],
    default="",
)
is_micro = (visits["micro"] != "").values

# Label of the HEALPix pixel containing the visit centre (RA, Dec in degrees)
pix = hp.ang2pix(NSIDE, visits["fieldRA"].values, visits["fieldDec"].values, lonlat=True)
visits["label"] = labels[pix]

# Index of the filter (0..5); -1 if the filter is not in BANDS
band_idx = visits["band"].map({b: i for i, b in enumerate(BANDS)}).fillna(-1).astype(int).values
valid_band = band_idx >= 0
print(f"Visits in a filter outside {BANDS}: {(~valid_band).sum()}")
print(visits["band"].value_counts().to_string())

In [ ]:
sky_label = visits["label"].values
not_micro = ~is_micro

masks = {}
masks["All visits"] = np.ones(N_ALL, dtype=bool)
masks["WFD"] = not_micro & np.isin(sky_label, wfd_labels)
masks["WFD low dust"] = not_micro & np.isin(sky_label, WFD_REGIONS["low dust"])
masks["DDF"] = visits["micro"].values == "DDF"
masks["Mini-surveys (total)"] = not_micro & np.isin(sky_label, mini_labels)
for name, labs in MINI_SURVEYS.items():
    masks[name] = not_micro & np.isin(sky_label, labs)
for name in ["RGES", "TwilightNEO", "ToO"]:
    masks[name] = visits["micro"].values == name
masks["Micro-surveys without DDF (total)"] = is_micro & ~masks["DDF"]
masks["Unassigned"] = not_micro & ~masks["WFD"] & ~masks["Mini-surveys (total)"]

n_sum = sum(
    masks[n].sum()
    for n in ["WFD", "DDF", "Mini-surveys (total)", "Micro-surveys without DDF (total)", "Unassigned"]
)
print(f"WFD + DDF + mini + micro without DDF + unassigned = {n_sum}, all visits = {N_ALL}")
assert n_sum == N_ALL, "a visit is counted twice or not at all"
print(f'Unassigned visits: {masks["Unassigned"].sum()}')

# Display order of the categories
CATEGORIES = (
    ["All visits", "WFD", "WFD low dust", "DDF", "Mini-surveys (total)"]
    + mini_names
    + ["RGES", "TwilightNEO", "ToO", "Micro-surveys without DDF (total)"]
)
CATEGORIES = [c for c in CATEGORIES if c in masks]

# Categories with too few visits are removed from the study
n_vis = {c: int((masks[c] & valid_band).sum()) for c in CATEGORIES}
for c in CATEGORIES:
    if n_vis[c] < MIN_VISITS_TOTAL:
        print(f"Category removed (too few visits): {c}, {n_vis[c]} visits")
CATEGORIES = [c for c in CATEGORIES if n_vis[c] >= MIN_VISITS_TOTAL]
print("Categories studied:", CATEGORIES)

## 2. Number of visits per pointing with the MAF `NVisits` metric

The metric and the slicer are the ones of cell 9 of `03_fbs5.3.6/Footprint_v2026-10-07.ipynb`:

```python
metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")
slicer = maf.HealpixSlicer(nside=64)
```

A **pointing** is a HEALPix pixel (`NSIDE_POINTING`); its number of visits is the number of visits that the `HealpixSlicer` attaches to the pixel (field of view of radius 1.75 deg around `fieldRA`, `fieldDec`; depending on the installed version, the slicer may use the LSST camera footprint, with `rotSkyPos`, instead of a plain disc).

**How the categories are handled.** A SQL constraint cannot select the categories defined by the position of the visit in the scheduler map (`WFD`, mini-surveys). The slicer is therefore set up **once on all visits** (like the `constraint = None` of cell 9). For each pixel, the list of visits attached to the pixel by the slicer is split by category and by filter, and `metric.run` is applied to each sub-selection. The pixel-to-visit association of a spatial slicer does not depend on the other visits, so this gives the same numbers as one MAF run per category and per filter with the corresponding visits.

For a given category, the pointings are the pixels that received **at least one visit of the category** in any filter. A pointing never observed in a given filter has 0 visit in this filter: it is included in the histogram of this filter (first bin), in the mean and in the median.

The loop over the pixels takes a few minutes; the maps are saved in `DATA_DIR` (`.npz`) and read back when `USE_CACHE = True`.

In [ ]:
# Same metric and slicer as in cell 9 of 03_fbs5.3.6/Footprint_v2026-10-07.ipynb
metric = maf.CountMetric("observationStartMJD", metric_name="NVisits")

slicer = maf.HealpixSlicer(nside=NSIDE_POINTING)
print("maf.HealpixSlicer ::  radius = ", slicer.radius, " use_camera = ", slicer.use_camera)
npix = hp.nside2npix(NSIDE_POINTING)
POINTING_DESC = f"MAF NVisits on HealpixSlicer nside={NSIDE_POINTING}"
print("Pointing definition:", POINTING_DESC)

# Visits given to the slicer: numpy record array with the columns it needs
MAF_COLS = sorted(
    {"fieldRA", "fieldDec", "rotSkyPos", "observationStartMJD"} | set(getattr(slicer, "columns_needed", []))
)
print("Columns given to the MAF slicer:", MAF_COLS)
missing = [c for c in MAF_COLS if c not in visits.columns]
assert not missing, f"columns missing in the visits table, add them to the SQL query: {missing}"
sim_data = visits[MAF_COLS].to_records(index=False)

# Boolean masks of the categories, restricted to the six filters
cat_masks = np.array([masks[c] & valid_band for c in CATEGORIES])  # (n_categories, n_visits)

cache_file = os.path.join(DATA_DIR, f"{opsim_name}_maf_nvisits_nside{NSIDE_POINTING}.npz")
nv = None  # (n_categories, npix, 6): NVisits per pixel, per filter, per category
if USE_CACHE and os.path.exists(cache_file):
    cache = np.load(cache_file)
    if list(cache["categories"]) == list(CATEGORIES) and cache["nv"].shape[1] == npix:
        nv = cache["nv"]
        print("NVisits maps read from", cache_file)

if nv is None:
    slicer.setup_slicer(sim_data)
    assert slicer.nslice == npix, f"unexpected number of slices: {slicer.nslice} instead of {npix}"
    nv = np.zeros((len(CATEGORIES), npix, len(BANDS)), dtype=np.int32)
    for ipix, sl in enumerate(slicer):
        idxs = np.asarray(sl["idxs"], dtype=int)  # visits attached to this pixel by the slicer
        if len(idxs) > 0:
            for c in range(len(CATEGORIES)):
                sel = idxs[cat_masks[c, idxs]]
                if len(sel) == 0:
                    continue
                b = band_idx[sel]
                for k in range(len(BANDS)):
                    sel_k = sel[b == k]
                    if len(sel_k) > 0:
                        nv[c, ipix, k] = metric.run(sim_data[sel_k], slice_point=sl["slice_point"])
        if ipix % 10000 == 0:
            print(f"pixel {ipix} / {npix}")
    np.savez_compressed(cache_file, nv=nv, categories=np.array(CATEGORIES))
    print("NVisits maps saved in", cache_file)

vpp = {}  # category -> array (n_pointings, 6)
for c, cat in enumerate(CATEGORIES):
    vpp[cat] = nv[c][nv[c].sum(axis=1) > 0]  # keep the pointings with at least one visit
    print(
        f"{cat:35s} {len(vpp[cat]):8d} pointings, {vpp[cat].sum(axis=1).mean():8.1f} visits per pointing (all filters)"
    )

### Check against cell 9 of `Footprint_v2026-10-07`

MAF is run exactly as in cell 9 (`MetricBundle` + `MetricBundleGroup` on the database, no constraint, all visits and all filters). The sum over the filters of the map of the `All visits` category above must be identical to this map. The run is skipped with `RUN_CELL9_CHECK = False`.

In [ ]:
if RUN_CELL9_CHECK:
    # Same calls as in cell 9 (all visits, all filters, no constraint)
    metric9 = maf.CountMetric("observationStartMJD", metric_name="NVisits")
    slicer9 = maf.HealpixSlicer(nside=NSIDE_POINTING)
    bundle9 = maf.MetricBundle(metric9, slicer9, None, run_name=opsim_name)
    g9 = maf.MetricBundleGroup({"nvisits": bundle9}, opsdb, out_dir=DATA_DIR, verbose=True)
    g9.run_all()
    nv_cell9 = bundle9.metric_values.filled(0)

    nv_here = nv[CATEGORIES.index("All visits")].sum(axis=1)
    diff = nv_here - nv_cell9
    print(
        f"cell 9 : {int((nv_cell9 > 0).sum())} pixels with visits, max {nv_cell9.max():.0f}, mean {nv_cell9[nv_cell9 > 0].mean():.1f}"
    )
    print(
        f"here   : {int((nv_here > 0).sum())} pixels with visits, max {nv_here.max():.0f}, mean {nv_here[nv_here > 0].mean():.1f}"
    )
    print(
        f"pixels with a different number of visits: {int((diff != 0).sum())}, max |difference| = {np.abs(diff).max():.0f}"
    )

## 3. Mean and median number of visits per pointing

Table of the numbers given in the legends of the figures: number of pointings of the category, then mean and median number of visits per pointing in each filter. It is saved in `DATA10ter_FilterBalance/` (`..._visits_per_pointing_stats.csv`) and can be compared with the table of `10bis_FilterBalance`.

Caution: MAF counts a visit in every pixel attached to its field of view (about 11 pixels at `nside=64`), so the numbers are visits per **sky position**, not visits per field centre.

In [ ]:
rows = {}
for cat in CATEGORIES:
    v = vpp[cat]
    row = {"N pointings": len(v)}
    for i, b in enumerate(BANDS):
        row[f"mean_{b}"] = v[:, i].mean()
    for i, b in enumerate(BANDS):
        row[f"median_{b}"] = np.median(v[:, i])
    rows[cat] = row
stats_table = pd.DataFrame(rows).T
stats_table["N pointings"] = stats_table["N pointings"].astype(int)
stats_table.round(2).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_pointing_stats.csv"))

fmt = {"N pointings": "{:,.0f}"}
fmt.update({c: "{:.1f}" for c in stats_table.columns if c != "N pointings"})
stats_table.style.format(fmt)

In [ ]:
def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=150, bbox_inches="tight")


def grid_axes(n, ncols=3, sharex=True):
    # Grid of n panels (3 columns); the extra panels are hidden
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.2 * nrows), sharex=sharex, squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


def plot_pointing_hists(stacked):
    # One panel per category; one histogram per filter, with the colours of the filters
    n = len(CATEGORIES)
    fig, axes = grid_axes(n, sharex=False)
    colors = [BAND_COLORS[b] for b in BANDS]
    for ax, cat in zip(axes, CATEGORIES):
        v = vpp[cat]
        log_x = cat in LOG_X_CATEGORIES
        # x range: XLIM[cat] = (xmin, xmax) if given, else from 0 (1 on a log axis) to a percentile of the counts
        xmin, xmax = XLIM.get(cat, (None, None))
        if xmax is None:
            xmax = np.percentile(v, XMAX_PERCENTILE) if XMAX_PERCENTILE else v.max()
        vmax = max(int(np.ceil(xmax)), 2)
        vmin = int(xmin) if xmin is not None else (1 if log_x else 0)
        # Same bins for the six filters; integer-centred bins when the counts are small
        if log_x:
            bins = np.logspace(np.log10(max(vmin, 1)), np.log10(vmax), N_BINS + 1)
        elif vmax - vmin < N_BINS:
            bins = np.arange(vmin - 0.5, vmax + 1.5)
        else:
            bins = np.linspace(vmin, vmax, N_BINS + 1)
        n_out = int((v > vmax).sum())
        if n_out:
            print(f"{cat}: {n_out} (pointing, filter) entries above xmax={vmax} are not shown")
        labels = [
            f"{b}: aver={v[:, i].mean():.0f}, med={np.median(v[:, i]):.0f}" for i, b in enumerate(BANDS)
        ]
        if stacked:
            ax.hist(
                [v[:, i] for i in range(len(BANDS))],
                bins=bins,
                stacked=True,
                color=colors,
                label=labels,
                edgecolor="none",
            )
        else:
            for i in range(len(BANDS)):
                # filled histogram (transparent) + outline, the legend uses the outline
                ax.hist(v[:, i], bins=bins, histtype="stepfilled", color=colors[i], alpha=0.30)
                ax.hist(v[:, i], bins=bins, histtype="step", color=colors[i], lw=1.8, label=labels[i])
        ax.set_title(cat)
        ax.grid(alpha=0.3)
        if log_x:
            ax.set_xscale("log")
        ax.set_xlim(bins[0], bins[-1])
        if LOG_Y:
            ax.set_yscale("log")
        ax.legend(
            title=f"visits per pointing ({len(v):,} pointings)",
            fontsize=10,
            title_fontsize=10,
            loc="best",
            framealpha=0.85,
        )
    for k, ax in enumerate(axes[:n]):
        if k % 3 == 0:
            ax.set_ylabel("Number of pointings")
        if k + 3 >= n:
            ax.set_xlabel("Number of visits per pointing")
    kind = "stacked" if stacked else "non-stacked"
    fig.suptitle(f"Visits per pointing, {kind} histograms - {opsim_name}\n(pointing = {POINTING_DESC})")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return fig

## 4. Histograms of the number of visits per pointing

### Figure 1 - Non-stacked histograms (`<run>_visits_per_pointing_hist.png`)

One panel per category, one histogram per filter (same colour code as in `10_FilterBalance`). Each pointing contributes once to the histogram of each filter. The legend gives the **mean** (`aver`) and the **median** (`med`) number of visits per pointing in each filter, and the number of pointings of the category. Each panel has its own horizontal scale; the bins are the same for the six filters of a panel.

In [ ]:
fig1 = plot_pointing_hists(stacked=False)
save_fig(fig1, "visits_per_pointing_hist")

### Figure 2 - Stacked histograms (`<run>_visits_per_pointing_hist_stacked.png`)

Same panels and same bins as Figure 1, but the histograms of the six filters are **stacked** (order u, g, r, i, z, y from the bottom): in each bin, the total height is the number of pointings, whatever the filter, with a given number of visits, and the coloured segments give the contribution of each filter. The legend is the same as in Figure 1 (mean and median per filter).

In [ ]:
fig2 = plot_pointing_hists(stacked=True)
save_fig(fig2, "visits_per_pointing_hist_stacked")

## 5. Comparison with `10bis_FilterBalance`

- The categories, the masks, the colours, the legends and the figures are the same as in `10bis_FilterBalance`; only the counting of the visits per pointing differs (MAF `NVisits` here, KD-tree on a disc of radius 1.75 deg in `10bis`).
- To compare the numbers, read the two tables `DATA10bis_FilterBalance/<run>_visits_per_pointing_stats.csv` and `DATA10ter_FilterBalance/<run>_visits_per_pointing_stats.csv` (same rows and columns).
- A systematic difference between the two means that the MAF slicer does not attach to a pixel exactly the visits of a disc of radius 1.75 deg (for instance because of the camera footprint); the check of section 2 shows that the `All visits` map is the one of cell 9 of `Footprint_v2026-10-07`.